# Final four-class ResNet-18 experiment (Kaggle GPU)

This notebook audits and deduplicates image bytes in memory, makes a stratified split, trains an ImageNet-pretrained ResNet-18 for 10 epochs, selects by validation accuracy, and reports metrics only on the held-out test set. Dataset files are read-only. Outputs are written to `/kaggle/working/outputs`.


## Dataset provenance note
The 90 `lab_external_XXXX.png` files have matching SHA-256, source IDs, PDF filenames, and page numbers in the project expansion report. The original GDC PDFs were not found locally, so those image-to-page mappings were not independently checked against the PDF bytes. The run below uses the supplied current dataset as requested; interpret its Lab_Report results with that provenance limitation in mind.


In [ ]:
# Cell 1 - Dependencies, imports, configuration, reproducible seeds
# Kaggle images normally include these packages. If missing, uncomment the install line.
# %pip install -q torch torchvision scikit-learn pandas matplotlib pillow
import os, json, hashlib, random, platform
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix)

SEED = 42
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
EXPECTED = {'Discharge_Summary': 500, 'Lab_Report': 516, 'Prescription': 129, 'Non_Medical': 700}
CLASS_DIRS = {'Discharge_Summary':'discharge_summaries', 'Lab_Report':'lab_reports',
              'Prescription':'handwritten_prescription', 'Non_Medical':'non_medical'}
CLASSES = list(CLASS_DIRS)
EXTENSIONS = {'.png','.jpg','.jpeg','.webp','.bmp','.tif','.tiff'}
OUT = Path('/kaggle/working/outputs')
OUT.mkdir(parents=True, exist_ok=True)

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(SEED)
print('Seed:', SEED)
print('PyTorch:', torch.__version__)
print('Python:', platform.python_version())
print('Output directory:', OUT)


In [ ]:
# Cell 2 - Automatically discover dataset root under Kaggle input
# Required child directory names are used only to locate class folders; labels come from this explicit mapping.
search_roots = [Path('/kaggle/input')]
if Path('/kaggle/working').exists(): search_roots.append(Path('/kaggle/working'))
candidates = []
for search_root in search_roots:
    if not search_root.exists(): continue
    for d in search_root.rglob(CLASS_DIRS[CLASSES[0]]):
        if d.is_dir():
            root = d.parent
            if all((root / dirname).is_dir() for dirname in CLASS_DIRS.values()):
                candidates.append(root.resolve())
candidates = sorted(set(candidates), key=lambda p: (len(p.parts), str(p)))
if not candidates:
    raise FileNotFoundError('Could not find one root containing all four expected class folders. Attach/upload the dataset to Kaggle.')
# If multiple copies exist, choose the root with the expected total image count; otherwise require manual disambiguation.
def count_root(root):
    return {label: sum(1 for p in (root / dirname).rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONS)
            for label, dirname in CLASS_DIRS.items()}
exact = [r for r in candidates if count_root(r) == EXPECTED]
if len(exact) == 1: DATASET_ROOT = exact[0]
elif len(candidates) == 1: DATASET_ROOT = candidates[0]
else:
    raise RuntimeError('Multiple possible dataset roots found; detach duplicates or set DATASET_ROOT explicitly. Candidates: ' + str([str(x) for x in candidates]))
print('Dataset root:', DATASET_ROOT)


In [ ]:
# Cell 3 - Dataset audit and required count safety stop
paths_by_class = {}
counts = {}
for label, dirname in CLASS_DIRS.items():
    paths = sorted(p for p in (DATASET_ROOT / dirname).rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONS)
    paths_by_class[label] = paths
    counts[label] = len(paths)
print('==================================================')
print('FINAL DATASET AUDIT')
print('==================================================')
for label in CLASSES: print(f'{label}: {counts[label]}')
print('Total:', sum(counts.values()))
print('Dataset root:', DATASET_ROOT)
if counts != EXPECTED:
    raise RuntimeError(f'Count discrepancy; stopping before training. Expected {EXPECTED}, found {counts}')
# Confirm images decode; this is read-only.
invalid = []
for label, paths in paths_by_class.items():
    for p in paths:
        try:
            with Image.open(p) as im: im.verify()
        except Exception as e: invalid.append({'file':str(p),'error':str(e)})
if invalid:
    raise RuntimeError(f'{len(invalid)} unreadable image(s); stopping before training. First examples: {invalid[:5]}')
print('All images pass PIL integrity verification.')


In [ ]:
# Cell 4 - SHA-256 exact duplicate detection (training list only; originals remain untouched)
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()
records, seen = [], {}
duplicate_rows = []
for label in CLASSES:
    for p in paths_by_class[label]:
        digest = sha256_file(p)
        if digest in seen:
            prior = seen[digest]
            if prior['label'] != label:
                raise RuntimeError(f'Exact duplicate has conflicting class labels: {prior["label"]} vs {label}: {p}')
            duplicate_rows.append({'duplicate_file':str(p),'kept_file':prior['path'],'sha256':digest,'label':label})
        else:
            row = {'path':str(p),'label':label,'sha256':digest}
            seen[digest] = row; records.append(row)
DEDUP_REPORT = {'images_before_deduplication':sum(counts.values()),
                'duplicate_occurrences':len(duplicate_rows),'unique_images_used':len(records),
                'duplicates':duplicate_rows}
(OUT/'deduplication_report.json').write_text(json.dumps(DEDUP_REPORT,indent=2),encoding='utf-8')
print(json.dumps({k:v for k,v in DEDUP_REPORT.items() if k!='duplicates'},indent=2))


In [ ]:
# Cell 5 - Stratified 80/10/10 split, performed after deduplication
all_df = pd.DataFrame(records)
train_df, holdout_df = train_test_split(all_df, test_size=0.20, random_state=SEED, stratify=all_df['label'])
val_df, test_df = train_test_split(holdout_df, test_size=0.50, random_state=SEED, stratify=holdout_df['label'])
train_df=train_df.reset_index(drop=True); val_df=val_df.reset_index(drop=True); test_df=test_df.reset_index(drop=True)
split_counts = {s:{c:int((df.label==c).sum()) for c in CLASSES}
                for s,df in [('train',train_df),('validation',val_df),('test',test_df)]}
print('Class | Total(unique) | Train | Validation | Test')
for c in CLASSES:
    print(f'{c} | {int((all_df.label==c).sum())} | {split_counts["train"][c]} | {split_counts["validation"][c]} | {split_counts["test"][c]}')
print('Split counts:', json.dumps(split_counts))


In [ ]:
# Cell 6 - Dataset distribution and split distribution charts
fig, ax = plt.subplots(figsize=(9,5)); vals=[counts[c] for c in CLASSES]
ax.bar(CLASSES, vals, color='#3973ac'); ax.set_title('Dataset Distribution Before Deduplication')
ax.set_ylabel('Images'); ax.set_xlabel('Class'); ax.tick_params(axis='x',rotation=15); fig.tight_layout()
fig.savefig(OUT/'dataset_distribution.png',dpi=200); plt.show(); plt.close(fig)
plot_df=pd.DataFrame({'Train':[split_counts['train'][c] for c in CLASSES],
                      'Validation':[split_counts['validation'][c] for c in CLASSES],
                      'Test':[split_counts['test'][c] for c in CLASSES]},index=CLASSES)
ax=plot_df.plot(kind='bar',figsize=(10,5),color=['#3973ac','#e09f3e','#548c55'])
ax.set_title('Stratified Split Distribution'); ax.set_ylabel('Unique images'); ax.set_xlabel('Class')
plt.xticks(rotation=15); plt.tight_layout(); plt.savefig(OUT/'dataset_split_distribution.png',dpi=200); plt.show(); plt.close()


In [ ]:
# Cell 7 - Image transforms
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.15,contrast=0.15,saturation=0.15,hue=0.05),
    transforms.RandomAffine(degrees=0,translate=(0.05,0.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
print('Training and evaluation transforms configured.')


In [ ]:
# Cell 8 - Datasets and DataLoaders
class ImageRows(Dataset):
    def __init__(self, frame, transform): self.frame=frame.reset_index(drop=True); self.transform=transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row=self.frame.iloc[i]
        with Image.open(row.path) as im: x=self.transform(im.convert('RGB'))
        return x, CLASSES.index(row.label)

pin = torch.cuda.is_available()
train_loader=DataLoader(ImageRows(train_df,train_transform),batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=pin)
val_loader=DataLoader(ImageRows(val_df,eval_transform),batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=pin)
test_loader=DataLoader(ImageRows(test_df,eval_transform),batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=pin)
print('Train / validation / test batches:',len(train_loader),len(val_loader),len(test_loader))


In [ ]:
# Cell 9 - Pretrained ResNet-18 and runtime details
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)
print('GPU name:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
print('PyTorch version:',torch.__version__)
weights=models.ResNet18_Weights.DEFAULT
model=models.resnet18(weights=weights)
model.fc=nn.Linear(model.fc.in_features,len(CLASSES))
model=model.to(device)
print('Model: torchvision ResNet-18, ImageNet DEFAULT pretrained weights; classes:',CLASSES)


In [ ]:
# Cell 10 - Class weights from training split only
train_labels=train_df['label'].to_numpy()
weight_values=compute_class_weight(class_weight='balanced',classes=np.array(CLASSES),y=train_labels)
class_weights=torch.tensor(weight_values,dtype=torch.float32,device=device)
print('Class weights (training split only):', {c:float(w) for c,w in zip(CLASSES,weight_values)})
criterion=nn.CrossEntropyLoss(weight=class_weights)
optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)


In [ ]:
# Cell 11 - Train for 10 epochs; retain the best validation-accuracy checkpoint
history=[]; best_val_acc=-1.0; best_epoch=None; best_checkpoint=OUT/'medical_document_resnet18_v2_final.pth'
for epoch in range(1,EPOCHS+1):
    model.train(); train_loss_sum=0.0; train_correct=0; train_n=0
    for xb,yb in train_loader:
        xb=xb.to(device,non_blocking=True); yb=yb.to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True); logits=model(xb); loss=criterion(logits,yb)
        loss.backward(); optimizer.step()
        train_loss_sum += loss.item()*yb.size(0); train_correct += (logits.argmax(1)==yb).sum().item(); train_n += yb.size(0)
    model.eval(); val_loss_sum=0.0; val_correct=0; val_n=0
    with torch.no_grad():
        for xb,yb in val_loader:
            xb=xb.to(device,non_blocking=True); yb=yb.to(device,non_blocking=True)
            logits=model(xb); loss=criterion(logits,yb)
            val_loss_sum += loss.item()*yb.size(0); val_correct += (logits.argmax(1)==yb).sum().item(); val_n += yb.size(0)
    row={'epoch':epoch,'train_loss':train_loss_sum/train_n,'train_accuracy':train_correct/train_n,
         'validation_loss':val_loss_sum/val_n,'validation_accuracy':val_correct/val_n}
    history.append(row)
    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'Train Loss: {row["train_loss"]:.6f}'); print(f'Train Accuracy: {row["train_accuracy"]:.6f}')
    print(f'Validation Loss: {row["validation_loss"]:.6f}'); print(f'Validation Accuracy: {row["validation_accuracy"]:.6f}')
    if row['validation_accuracy'] > best_val_acc:
        best_val_acc=row['validation_accuracy']; best_epoch=epoch
        torch.save({'state_dict':model.state_dict(),'class_mapping':{c:i for i,c in enumerate(CLASSES)},
                    'best_epoch':best_epoch,'best_validation_accuracy':best_val_acc},best_checkpoint)
        best_info={'best_epoch':best_epoch,'best_validation_accuracy':best_val_acc,
          'training_configuration':{'architecture':'ResNet-18','weights':'ResNet18_Weights.DEFAULT',
            'optimizer':'Adam','learning_rate':LEARNING_RATE,'epochs':EPOCHS,'batch_size':BATCH_SIZE,'seed':SEED,
            'loss':'weighted CrossEntropyLoss; weights computed from training set only'},
          'class_mapping':{c:i for i,c in enumerate(CLASSES)}}
        (OUT/'best_model_info.json').write_text(json.dumps(best_info,indent=2),encoding='utf-8')
print('Best epoch:',best_epoch,'Best validation accuracy:',best_val_acc)


In [ ]:
# Cell 12 - Training history table and CSV
history_df=pd.DataFrame(history)
history_df.to_csv(OUT/'training_history.csv',index=False)
display(history_df)


In [ ]:
# Cell 13 - Accuracy curve from actual training_history.csv
history_df=pd.read_csv(OUT/'training_history.csv')
fig,ax=plt.subplots(figsize=(8,5)); ax.plot(history_df.epoch,history_df.train_accuracy,marker='o',label='Train Accuracy')
ax.plot(history_df.epoch,history_df.validation_accuracy,marker='o',label='Validation Accuracy')
ax.set(xlabel='Epoch',ylabel='Accuracy',title='Training and Validation Accuracy'); ax.set_xticks(history_df.epoch); ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
fig.savefig(OUT/'accuracy_curve.png',dpi=180); fig.savefig(OUT/'accuracy_curve_300dpi.png',dpi=300); plt.show(); plt.close(fig)


In [ ]:
# Cell 14 - Loss curve from actual training_history.csv
fig,ax=plt.subplots(figsize=(8,5)); ax.plot(history_df.epoch,history_df.train_loss,marker='o',label='Train Loss')
ax.plot(history_df.epoch,history_df.validation_loss,marker='o',label='Validation Loss')
ax.set(xlabel='Epoch',ylabel='Loss',title='Training and Validation Loss'); ax.set_xticks(history_df.epoch); ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
fig.savefig(OUT/'loss_curve.png',dpi=180); fig.savefig(OUT/'loss_curve_300dpi.png',dpi=300); plt.show(); plt.close(fig)


In [ ]:
# Cell 15 - Load the best validation-accuracy model before test evaluation
checkpoint=torch.load(best_checkpoint,map_location=device)
model.load_state_dict(checkpoint['state_dict']); model.eval()
print('Loaded checkpoint:',best_checkpoint)
print('Best epoch:',checkpoint['best_epoch'],'Best validation accuracy:',checkpoint['best_validation_accuracy'])


In [ ]:
# Cell 16 - Held-out test inference only
all_true=[]; all_pred=[]
with torch.no_grad():
    for xb,yb in test_loader:
        logits=model(xb.to(device,non_blocking=True))
        all_true.extend(yb.numpy().tolist()); all_pred.extend(logits.argmax(1).cpu().numpy().tolist())
print('Held-out test samples evaluated:',len(all_true))


In [ ]:
# Cell 17 - Test metrics and classification report
acc=accuracy_score(all_true,all_pred)
mp,mr,mf,_=precision_recall_fscore_support(all_true,all_pred,labels=list(range(len(CLASSES))),average='macro',zero_division=0)
wp,wr,wf,_=precision_recall_fscore_support(all_true,all_pred,labels=list(range(len(CLASSES))),average='weighted',zero_division=0)
test_metrics={'accuracy':float(acc),'macro_precision':float(mp),'macro_recall':float(mr),'macro_f1':float(mf),
               'weighted_precision':float(wp),'weighted_recall':float(wr),'weighted_f1':float(wf),'test_sample_count':len(all_true)}
print(json.dumps(test_metrics,indent=2))
report_txt=classification_report(all_true,all_pred,labels=list(range(len(CLASSES))),target_names=CLASSES,zero_division=0,digits=4)
report_dict=classification_report(all_true,all_pred,labels=list(range(len(CLASSES))),target_names=CLASSES,zero_division=0,output_dict=True)
(OUT/'test_metrics.json').write_text(json.dumps(test_metrics,indent=2),encoding='utf-8')
(OUT/'classification_report.txt').write_text(report_txt,encoding='utf-8')
pd.DataFrame(report_dict).T.to_csv(OUT/'classification_report.csv',index_label='class')
print(report_txt)


In [ ]:
# Cell 18 - Confusion matrix, fixed class order
cm=confusion_matrix(all_true,all_pred,labels=list(range(len(CLASSES))))
print('Class order:',CLASSES); print(cm)
pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(OUT/'confusion_matrix.csv',index_label='true\predicted')
fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(cm,cmap='Blues'); fig.colorbar(im,ax=ax)
ax.set(xticks=range(len(CLASSES)),yticks=range(len(CLASSES)),xticklabels=CLASSES,yticklabels=CLASSES,
       xlabel='Predicted label',ylabel='True label',title='Held-out Test Confusion Matrix')
plt.setp(ax.get_xticklabels(),rotation=25,ha='right',rotation_mode='anchor')
threshold=cm.max()/2 if cm.size else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]): ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>threshold else 'black',fontsize=12)
fig.tight_layout(); fig.savefig(OUT/'confusion_matrix.png',dpi=220); fig.savefig(OUT/'confusion_matrix_300dpi.png',dpi=300); plt.show(); plt.close(fig)


In [ ]:
# Cell 19 - Per-class precision, recall and F1 graph (held-out test)
per_class=report_dict
metric_df=pd.DataFrame({m:[per_class[c][m] for c in CLASSES] for m in ['precision','recall','f1-score']},index=CLASSES)
ax=metric_df.plot(kind='bar',figsize=(10,5),ylim=(0,1),color=['#3973ac','#e09f3e','#548c55'])
ax.set_title('Per-Class Metrics on Held-out Test Set'); ax.set_ylabel('Score'); ax.set_xlabel('Class'); plt.xticks(rotation=15); plt.tight_layout()
plt.savefig(OUT/'per_class_metrics.png',dpi=240); plt.show(); plt.close()


In [ ]:
# Cell 20 - Final results summary graphic, values sourced from test_metrics
summary_lines=[('Accuracy',acc),('Macro Precision',mp),('Macro Recall',mr),('Macro F1',mf)]
fig,ax=plt.subplots(figsize=(9,5)); ax.axis('off'); ax.text(.5,.9,'ResNet-18 ? Final Test Results',ha='center',va='center',fontsize=22,weight='bold')
for i,(label,value) in enumerate(summary_lines): ax.text(.12,.72-i*.15,f'{label}: {value*100:.2f}%',ha='left',fontsize=18)
fig.tight_layout(); fig.savefig(OUT/'final_results_summary.png',dpi=240,bbox_inches='tight'); plt.show(); plt.close(fig)


In [ ]:
# Cell 21 - Complete machine-readable and human-readable experiment reports
report={
 'dataset':{'root':str(DATASET_ROOT),'class_mapping':CLASS_DIRS,'counts_before_deduplication':counts,
           'total_before_deduplication':sum(counts.values()),'unique_after_sha256_deduplication':len(records)},
 'deduplication':{k:v for k,v in DEDUP_REPORT.items() if k!='duplicates'},
 'split_counts':split_counts,'split_ratios':{'train':.8,'validation':.1,'test':.1},
 'seed':SEED,'class_weights_train_only':{c:float(w) for c,w in zip(CLASSES,weight_values)},
 'model_configuration':{'architecture':'torchvision.models.resnet18','weights':'ResNet18_Weights.DEFAULT',
  'num_classes':4,'optimizer':'Adam','learning_rate':LEARNING_RATE,'epochs':EPOCHS,'batch_size':BATCH_SIZE},
 'training_history':history,'best_epoch':int(best_epoch),'best_validation_accuracy':float(best_val_acc),
 'test_metrics':test_metrics,'per_class_metrics':{c:report_dict[c] for c in CLASSES},'confusion_matrix':cm.tolist(),
 'confusion_matrix_class_order':CLASSES,'checkpoint':str(best_checkpoint)}
(OUT/'resnet18_final_experiment_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
summary=['ResNet-18 Final Four-Class Experiment',f'Dataset root: {DATASET_ROOT}',f'Images before deduplication: {sum(counts.values())}',
 f'Exact duplicate occurrences: {len(duplicate_rows)}',f'Unique images: {len(records)}',f'Split counts: {json.dumps(split_counts)}',
 f'Best epoch: {best_epoch}',f'Best validation accuracy: {best_val_acc:.6f}',f'Test metrics: {json.dumps(test_metrics)}',
 'Per-class test metrics:',report_txt,'Confusion matrix class order: '+', '.join(CLASSES),str(cm.tolist())]
(OUT/'resnet18_final_experiment_report.txt').write_text('\n'.join(summary),encoding='utf-8')
print('Saved complete experiment JSON and text reports.')


In [ ]:
# Cell 22 - Final output inventory and PPT files
ppt_files=['accuracy_curve_300dpi.png','loss_curve_300dpi.png','confusion_matrix_300dpi.png','per_class_metrics.png',
'dataset_distribution.png','dataset_split_distribution.png','final_results_summary.png','classification_report.csv',
' training_history.csv'.strip(),'test_metrics.json']
print('PPT FILES GENERATED')
print('====================')
for name in ppt_files: print(name, 'OK' if (OUT/name).exists() else 'MISSING')
print('Exact Kaggle output directory:',OUT)
print('All outputs:',*[p.name for p in sorted(OUT.iterdir()) if p.is_file()],sep='\n')
